<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 12 </h1></center>
<center><h2> Coûts, latence et monitoring </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 12 · 20 min · ≈ 0,01 $ · prépare le Lab 7 et le chapitre 9 du projet**
> Appels en direct à la gateway, traces lues dans LangSmith (projet `ch9-replay`). **Sans gateway ni LangSmith, la démo se rabat sur les chiffres enregistrés.**

```
 coût     = tokens envoyés × prix  +  tokens reçus × prix
 latence  = durée d'un appel × nombre d'appels (un agent en fait plusieurs)
 à suivre : la VALEUR MAXIMALE, pas seulement la moyenne
```

In [1]:
import os
import time
from pathlib import Path
from statistics import mean

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# LangSmith region: the US endpoint must be set explicitly in the kernel.
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_TRACING"] = "false"  # the demo calls are not traced, only read

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])

# Public price of gpt-4o-mini, in dollars per million tokens (check your provider's price list)
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60
PROJECT = "ch9-replay"
print("modèle :", llm.model_name, "| projet LangSmith :", PROJECT)

modèle : gpt-4o-mini | projet LangSmith : ch9-replay


<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Un appel, mesuré </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Trois chiffres à chaque appel : **durée**, **tokens**, **coût**.
- Les tokens sont dans `usage_metadata` : **le fournisseur les compte pour nous**.

In [2]:
def measure(prompt, recorded=None):
    """Call the model once and return duration, tokens and cost.

    If the gateway is unreachable and `recorded` is given, the recorded numbers are returned instead.
    """
    start = time.perf_counter()
    try:
        response = llm.invoke(prompt)
    except Exception as error:
        if recorded is None:
            raise
        print("⚠️ gateway inaccessible :", type(error).__name__, "→ chiffres enregistrés")
        return {**recorded, "recorded": True}
    seconds = time.perf_counter() - start
    usage = response.usage_metadata
    cost = usage["input_tokens"] * PRICE_INPUT / 1e6 + usage["output_tokens"] * PRICE_OUTPUT / 1e6
    return {"seconds": seconds, "input": usage["input_tokens"], "output": usage["output_tokens"], "cost": cost}


PROMPT = """Répondez à la question à partir des extraits ci-dessous.

{context}

Question : {question}"""
QUESTION = "Mon fils de 8 ans a la grippe et j'ai deux enfants à charge. Combien de jours par an puis-je m'absenter pour le garder ?"
RECORDED_CALL = {"seconds": 1.0, "input": 1148, "output": 37, "cost": 0.00019}

fiche = Path("data/corpus/F151.md").read_text(encoding="utf-8")
call = measure(PROMPT.format(context=fiche, question=QUESTION), recorded=RECORDED_CALL)

print(f"Durée         : {call['seconds']:.1f} s")
print(f"Tokens envoyés: {call['input']}")
print(f"Tokens reçus  : {call['output']}")
print(f"Coût          : {call['cost']:.5f} $")
print("(valeurs enregistrées)" if call.get("recorded") else "(en direct : vos valeurs varient d'un appel à l'autre ; référence : 1148 tokens, 0,00019 $)")

Durée         : 0.9 s
Tokens envoyés: 1148
Tokens reçus  : 37
Coût          : 0.00019 $
(en direct : vos valeurs varient d'un appel à l'autre ; référence : 1148 tokens, 0,00019 $)


- **On paie surtout ce qu'on envoie** : le contexte pèse bien plus que la réponse.
- Un appel coûte **une fraction de centime**. Le problème apparaît **à l'échelle**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Plus de contexte, plus de tokens </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Même question, **de plus en plus de fiches** dans le contexte.
- Les tokens (donc le coût) suivent **la taille du contexte**.

In [3]:
corpus = sorted(Path("data/corpus").glob("F*.md"))
corpus = [path for path in corpus if path.stem != "F151"]

# Recorded numbers (fallback if the gateway is unreachable): fiches -> duration, tokens sent, cost
RECORDED_SIZES = {1: (0.8, 1148, 0.00020), 5: (1.1, 4403, 0.00071), 15: (1.4, 9905, 0.00154), 30: (1.7, 49595, 0.00748)}


def context_of(number_of_fiches):
    """The fiche that answers, plus other fiches, to make the context bigger."""
    texts = [fiche] + [path.read_text(encoding="utf-8") for path in corpus[:number_of_fiches - 1]]
    return "\n\n".join(texts)


rows = []
for number_of_fiches in (1, 5, 15, 30):
    seconds, tokens, cost = RECORDED_SIZES[number_of_fiches]
    recorded = {"seconds": seconds, "input": tokens, "output": None, "cost": cost}
    rows.append((number_of_fiches, measure(PROMPT.format(context=context_of(number_of_fiches), question=QUESTION), recorded=recorded)))

base_cost = rows[0][1]["cost"]
print(f"{'fiches':>7} {'tokens envoyés':>15} {'durée':>8} {'coût':>11} {'×':>6}")
for number_of_fiches, call in rows:
    print(f"{number_of_fiches:>7} {call['input']:>15} {call['seconds']:>6.1f} s {call['cost']:>9.5f} $ {call['cost'] / base_cost:>5.0f}×")
print("(valeurs enregistrées)" if any(call.get("recorded") for _, call in rows) else "(en direct : vos valeurs varient ; le rapport reste de l'ordre de la taille du contexte)")

 fiches  tokens envoyés    durée        coût      ×
      1            1148    1.1 s   0.00019 $     1×
      5            4403    1.3 s   0.00069 $     4×
     15            9905    1.6 s   0.00154 $     8×
     30           49595    3.4 s   0.00748 $    39×
(en direct : vos valeurs varient ; le rapport reste de l'ordre de la taille du contexte)


- **Le coût est proportionnel aux tokens envoyés.** 30 fiches coûtent **des dizaines de fois** plus qu'une seule, et la durée finit par monter aussi.
- **Plus de contexte n'est pas gratuit** : c'est aussi l'argument du RAG contre le « tout dans le prompt » (Part 5).
- Levier n° 1 : **envoyer moins** (bons passages, bonne taille de chunk).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Trois versions de l'assistant, mesurées sur 30 questions </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Les traces des 30 questions de chaque version sont dans **LangSmith**.
- On les relit **par programme** : `fetch_runs`, `summarize_runs`.

In [4]:
from src import monitoring

RECORDED = {
    "baseline": {"mean_latency_s": 1.3, "max_latency_s": 2.1, "mean_tokens": 1136, "mean_cost_usd": 0.00019, "errors": 0},
    "titled": {"mean_latency_s": 1.4, "max_latency_s": 3.0, "mean_tokens": 1529, "mean_cost_usd": 0.00025, "errors": 0},
    "agent": {"mean_latency_s": 6.9, "max_latency_s": 11.5, "mean_tokens": 19639, "mean_cost_usd": 0.00215, "errors": 0},
}

traces = {}
try:
    for version in RECORDED:
        traces[version] = monitoring.fetch_runs(PROJECT, version)
    summaries = {version: monitoring.summarize_runs(runs) for version, runs in traces.items()}
    source = "LangSmith, en direct"
except Exception as error:
    print("⚠️ LangSmith inaccessible :", type(error).__name__)
    traces = {}
    summaries = RECORDED
    source = "chiffres enregistrés"

print("Source :", source)
print(f"{'version':10} {'latence moy.':>13} {'latence max':>12} {'tokens':>8} {'coût':>11}")
for version, summary in summaries.items():
    print(f"{version:10} {summary['mean_latency_s']:>11} s {summary['max_latency_s']:>10} s {summary['mean_tokens']:>8} {summary['mean_cost_usd']:>9} $")

Source : LangSmith, en direct
version     latence moy.  latence max   tokens        coût
baseline           1.3 s        2.1 s     1136   0.00019 $
titled             1.4 s        3.0 s     1529   0.00025 $
agent              6.9 s       11.5 s    19639   0.00215 $


![](assets/12_versions.png)

- **L'agent coûte cher parce qu'il relit** : ×5 en durée, ×13 en tokens, ×9 en coût par rapport à `titled` (**chiffres de référence du chapitre 9**).
- Ce prix **ne se voyait pas** dans les mesures de qualité de la Part 9.
- **Le coût est une dimension de la comparaison**, au même titre que la qualité.
- La ligne `agent` vient d'une **3e exécution tracée**, pas des `agent_v2` de la Part 9.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Pourquoi l'agent relit : l'arbre de trace </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- Une trace = **l'arbre des étapes** d'une question : chaque tour du modèle, chaque outil.
- Question à plusieurs fiches (`q14`), version agent : **les tokens de chaque tour**.

In [5]:
RECORDED_TURNS = [353, 770, 1613, 2512, 8628]  # tokens of each model turn of q14 (recorded)

turns = RECORDED_TURNS
try:
    run = monitoring.find_run(traces["agent"], "q14")
    steps = monitoring.Client().list_runs(project_name=PROJECT, trace_id=str(run.trace_id), run_type="llm")
    turns = [step.total_tokens for step in sorted(steps, key=lambda step: step.start_time)]
    print("Source : LangSmith, en direct")
except Exception as error:
    print("⚠️ trace inaccessible, chiffres enregistrés :", type(error).__name__)

for number, tokens in enumerate(turns, start=1):
    print(f"tour {number} | {tokens:>6} tokens | " + "█" * (tokens // 250))
print(f"\nTotal : {sum(turns)} tokens pour UNE question (le tour 1 n'en coûtait que {turns[0]})")
print(f"Le dernier tour pèse {turns[-1] / turns[0]:.0f} fois le premier")

Source : LangSmith, en direct
tour 1 |    353 tokens | █
tour 2 |    770 tokens | ███
tour 3 |   1613 tokens | ██████
tour 4 |   2512 tokens | ██████████
tour 5 |   8628 tokens | ██████████████████████████████████

Total : 13876 tokens pour UNE question (le tour 1 n'en coûtait que 353)
Le dernier tour pèse 24 fois le premier


![](assets/12_agent_tokens.png)

- **À chaque tour, tout l'historique est renvoyé** : la question, les recherches, les fiches déjà lues.
- Les tokens **s'accumulent** : le dernier tour pèse **bien plus** que le premier (rapport calculé ci-dessus).
- Levier n° 2 : **limiter les tours**, lire moins de fiches, **résumer** l'historique.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. À 1 000 questions par jour </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- À l'échelle d'un exercice : **rien**. À l'échelle d'un service : **le rapport compte**.
- On projette le **coût moyen par question** sur le trafic.

In [6]:
QUESTIONS_PER_DAY = 1000
EXPENSIVE_MODEL = 10  # a model ten times more expensive

print(f"{'version':10} {'par jour':>10} {'par mois':>10} {'modèle ×10, par mois':>22}")
for version, summary in summaries.items():
    per_day = summary["mean_cost_usd"] * QUESTIONS_PER_DAY
    print(f"{version:10} {per_day:>8.2f} $ {per_day * 30:>8.2f} $ {per_day * 30 * EXPENSIVE_MODEL:>20.2f} $")

version      par jour   par mois   modèle ×10, par mois
baseline       0.19 $     5.70 $                57.00 $
titled         0.25 $     7.50 $                75.00 $
agent          2.15 $    64.50 $               645.00 $


- Chaîne : **environ 0,2 $ par jour**. Agent : **2,15 $**, soit **environ 9 fois plus**.
- Avec un modèle plus cher, **le rapport reste le même** : c'est lui qu'on retient, pas le prix du jour.
- **Un agent qui boucle se voit tout de suite dans la facture** → il faut une alerte.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 6. La latence : le maximum, pas la moyenne </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- L'utilisateur ne vit pas la moyenne : il vit **sa** question.
- On regarde la moyenne, le **dixième le plus lent** et le **maximum**.

In [7]:
BUDGET_SECONDS = 10  # what we accept to make a user wait

print(f"{'version':10} {'moyenne':>9} {'90e centile':>12} {'maximum':>9}   budget {BUDGET_SECONDS} s")
for version, summary in summaries.items():
    if traces:
        durations = sorted(monitoring.latency(run) for run in traces[version])
        slow = durations[int(0.9 * len(durations))]
        slow_text = f"{slow:>10.1f} s"
    else:
        slow_text = f"{'—':>12}"
    ok = summary["max_latency_s"] <= BUDGET_SECONDS
    print(f"{version:10} {summary['mean_latency_s']:>7} s {slow_text} {summary['max_latency_s']:>7} s   {'✅ tenu' if ok else '❌ dépassé'}")

version      moyenne  90e centile   maximum   budget 10 s
baseline       1.3 s        1.9 s     2.1 s   ✅ tenu
titled         1.4 s        1.8 s     3.0 s   ✅ tenu
agent          6.9 s       10.9 s    11.5 s   ❌ dépassé


![](assets/12_latency.png)

- L'agent : **6,9 s en moyenne**, mais **jusqu'à 11,5 s** pour les questions lentes.
- **La moyenne cache les pires cas** : on alerte sur le **maximum** (ou un centile élevé).
- Levier n° 3 : **streaming** (Part 7) : l'utilisateur voit la réponse arriver, l'attente paraît plus courte.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 7. Ce qu'on surveille en production </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| À surveiller | Alerte si… | Ce que ça révèle |
|---|---|---|
| **Latence maximale** | au-dessus du budget | agent trop de tours, modèle lent |
| **Coût par jour** | hausse brutale | **agent qui boucle**, contexte qui gonfle |
| **Taux d'erreur** | > 0 | gateway, limite de débit |
| **Taux de refus** | hausse brutale | recherche cassée, prompt modifié |
| **Texte envoyé au modèle** (gardé dans la trace) | — | **comprendre une mauvaise réponse après coup** |

In [8]:
RULES = [
    ("latence max ≤ 10 s", lambda s: s["max_latency_s"] <= 10),
    ("coût ≤ 0,001 $ par question", lambda s: s["mean_cost_usd"] <= 0.001),
    ("aucune erreur", lambda s: s["errors"] == 0),
]

print(f"{'version':10} " + " | ".join(name for name, _ in RULES))
for version, summary in summaries.items():
    marks = ["✅" if rule(summary) else "❌" for _, rule in RULES]
    print(f"{version:10} " + " | ".join(f"{mark:^{len(name)}}" for mark, (name, _) in zip(marks, RULES)))

version    latence max ≤ 10 s | coût ≤ 0,001 $ par question | aucune erreur
baseline           ✅          |              ✅              |       ✅      
titled             ✅          |              ✅              |       ✅      
agent              ❌          |              ❌              |       ✅      


- **Mêmes traces, deux usages** : comparer des versions hier, **surveiller** aujourd'hui.
- ⚠️ **Les questions de vrais salariés sont des données personnelles.** Réglable : `LANGSMITH_HIDE_INPUTS`, `LANGSMITH_TRACING_SAMPLING_RATE`, `LANGSMITH_TRACING=false`.
- **Sans le texte, plus de diagnostic** (le cas `q03` de la Part 9) : on décide **avec le juridique, avant la mise en production**.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Ce que la démo a montré | À retenir | Où vous le mettez en pratique |
|---|---|---|
| **Un appel mesuré** : durée, `usage_metadata`, coût | on paie surtout les tokens envoyés | **Lab 7** |
| **Coût vs taille du contexte** | proportionnel : envoyer moins | **Lab 7** |
| **Trois versions** : 1,4 s → 6,9 s, 0,00025 $ → 0,00215 $ | le coût est une dimension de la comparaison | **Projet, chapitre 9** (section 9.5) |
| **Arbre de trace de l'agent** | les tokens s'accumulent tour après tour | **Projet, chapitre 9** (section 9.6) |
| **1 000 questions par jour** | le rapport compte, pas le prix du jour | **Projet, chapitre 9** |
| **Latence max** : 11,5 s pour 6,9 s de moyenne | alerter sur le maximum | **Projet, chapitre 9** (section 9.8) |
| **Surveiller** : latence, coût, erreurs, refus | et décider de ce qu'on garde dans les traces | **Projet, chapitre 9** (section 9.8) |

- **Retenez** : mesurer **chaque appel** → comparer **les versions** → **surveiller le pire cas**.
- **Prochain pas : le Lab 7**, puis la Part 13 (production et CI).